# OPENBREWERY - LANDING NOTEBOOK

## 1.GENERAL SETTINGS

## 1.1 Load Variables

In [0]:
%run ./00-Config

## 1.2 Imports

In [0]:
import requests
import json
import time
import uuid

from datetime import datetime

from pyspark.sql.types import *

from pyspark.sql import functions as F

## 1.3 Parameters

In [0]:
dbutils.widgets.text(
    "base_url",
    "https://api.openbrewerydb.org/v1/breweries"
)

dbutils.widgets.text(
    "per_page",
    "200"
)

dbutils.widgets.text(
    "max_pages",
    "100"
)

BASE_URL = dbutils.widgets.get(
    "base_url"
)

PER_PAGE = int(
    dbutils.widgets.get("per_page")
)

MAX_PAGES = int(
    dbutils.widgets.get("max_pages")
)

REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_DELAY = 5

## 1.4 Notebook Variables

In [0]:
pipeline_name = "landing_openbrewery"

layer = "landing"

table_name = "openbrewery_landing"

execution_id = str(uuid.uuid4())

start_time = datetime.utcnow()

records_processed = 0

status = "RUNNING"

error_message = None

## 1.5 Metadata Execution

In [0]:
execution_timestamp = datetime.utcnow()

execution_date = execution_timestamp.strftime(
    "%Y-%m-%d"
)

execution_datetime = execution_timestamp.strftime(
    "%Y%m%d_%H%M%S"
)

landing_path = (
    f"{landing_volume_path}/"
    f"ingestion_date={execution_date}"
)

print("=" * 60)
print("STARTING LANDING INGESTION")
print(f"Execution ID: {execution_id}")
print(f"Execution Timestamp: {execution_timestamp}")
print(f"Landing Path: {landing_path}")
print("=" * 60)

# 2. Execution Start

## 2.1 Monitoring Log

In [0]:
spark.sql(f"""
INSERT INTO {monitoring_table}

VALUES (

    '{execution_id}',
    '{pipeline_name}',
    'LANDING',
    current_timestamp(),
    NULL,
    'RUNNING',
    0,
    NULL,
    NULL
)
""")

## 2.2 Create Landing Directory If Not Exists

In [0]:
dbutils.fs.mkdirs(landing_path)

## 2.3 Data Quality Metric Variables

In [0]:
empty_pages = 0

invalid_payloads = 0

api_errors = 0

small_payload_pages = 0

request_durations = []

## 2.4 HTTP Request Function

In [0]:
def make_request(url, params=None):

    global api_errors

    request_start = time.time()

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = requests.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            request_end = time.time()

            request_durations.append(
                round(request_end - request_start, 2)
            )

            if response.status_code == 200:

                return response.json()

            elif response.status_code == 429:

                api_errors += 1

                print(
                    f"Rate limit exceeded. "
                    f"Retry {attempt}/{MAX_RETRIES}"
                )

                time.sleep(RETRY_DELAY)

            elif response.status_code >= 500:

                api_errors += 1

                print(
                    f"Server error "
                    f"{response.status_code}. "
                    f"Retry {attempt}/{MAX_RETRIES}"
                )

                time.sleep(RETRY_DELAY)

            else:

                api_errors += 1

                raise Exception(
                    f"""
                    Request failed with
                    status code {response.status_code}
                    """
                )

        except requests.exceptions.Timeout:

            api_errors += 1

            print(
                f"Timeout occurred. "
                f"Retry {attempt}/{MAX_RETRIES}"
            )

            time.sleep(RETRY_DELAY)

        except requests.exceptions.ConnectionError:

            api_errors += 1

            print(
                f"Connection error. "
                f"Retry {attempt}/{MAX_RETRIES}"
            )

            time.sleep(RETRY_DELAY)

        except Exception as e:

            api_errors += 1

            print(
                f"Unexpected error: {str(e)}"
            )

            time.sleep(RETRY_DELAY)

    raise Exception(
        "Max retries exceeded"
    )

# 3. API Consumption

# 3.1 Paginated Ingestion

In [0]:
total_records = 0

total_pages = 0

try:

    for page in range(1, MAX_PAGES + 1):

        print(f"Processing page {page}")

        params = {
            "page": page,
            "per_page": PER_PAGE
        }

        data = make_request(
            BASE_URL,
            params=params
        )

        # ============================================
        # Validate invalid Payload
        # ============================================

        if not isinstance(data, list):

            invalid_payloads += 1

            continue
        
        # ============================================
        # Validate empty pages
        # ============================================
        if not data:

            empty_pages += 1

            print(
                f"No more records found at page {page}"
            )

            break

        # ============================================
        # validates pages with suspicious traffic
        # ============================================

        if len(data) < (PER_PAGE * 0.1):

            small_payload_pages += 1

        # ============================================
        # Ingest metadata
        # ============================================

        enriched_data = {

            "metadata": {

                "source": "openbrewerydb",

                "ingestion_timestamp_utc":
                    execution_timestamp.isoformat(),

                "page": page,

                "records": len(data)
            },

            "data": data
        }

        # ============================================
        # File name
        # ============================================

        file_name = (
            f"page_{str(page).zfill(4)}_"
            f"{execution_datetime}.json"
        )

        output_path = (
            f"{landing_path}/{file_name}"
        )

        # ============================================
        # Serialize json
        # ============================================

        json_content = json.dumps(
            enriched_data,
            ensure_ascii=False,
            indent=2
        )

        # ============================================
        # Save file
        # ============================================

        dbutils.fs.put(
            output_path,
            json_content,
            overwrite=True
        )

        print(f"Saved file: {output_path}")

        total_records += len(data)

        total_pages += 1

    status = "SUCCESS"

except Exception as e:

    status = "FAILED"

    error_message = str(e)

    raise

## 3.2 Data Quality Metrics

In [0]:
dq_data = [

    (
        execution_id,
        pipeline_name,
        layer,
        table_name,
        "empty_pages",
        empty_pages,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        table_name,
        "invalid_payloads",
        invalid_payloads,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        table_name,
        "api_errors",
        api_errors,
        datetime.utcnow()
    ),

    (
        execution_id,
        pipeline_name,
        layer,
        table_name,
        "small_payload_pages",
        small_payload_pages,
        datetime.utcnow()
    )
]

dq_df = spark.createDataFrame(
    dq_data,
    dq_schema
)

(
    dq_df.write

    .format("delta")

    .mode("append")

    .saveAsTable(
        data_quality_table
    )
)

## 3.3 Update Monitoring

In [0]:
spark.sql(f"""

UPDATE {monitoring_table}

SET

    end_time = current_timestamp(),

    status = '{status}',

    records_processed = {total_records},

    error_message = {f"'{error_message}'" if status == "FAILED" else "NULL"},

    created_at = current_timestamp()

WHERE execution_id = '{execution_id}'

""")

#4. EXECUTION SUMMARY

In [0]:
print("=" * 60)

print("INGESTION FINISHED")

print(f"Execution ID: {execution_id}")

print(f"Status: {status}")

print(f"Total Pages Processed: {total_pages}")

print(f"Total Records Ingested: {total_records}")

print(f"Empty Pages: {empty_pages}")

print(f"Invalid Payloads: {invalid_payloads}")

print(f"API Errors: {api_errors}")

print(f"Small Payload Pages: {small_payload_pages}")

print(f"Landing Path: {landing_path}")

print("=" * 60)